<a href="https://colab.research.google.com/github/abduyea/Career-Trends-Analyzer/blob/main/notebooks/Proejct_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Abdulfetah Adem

Spencer K

#Project Objective

Develop a data-driven tool that identifies the most in-demand technical and professional skills using real-world job posting data.

The tool will highlight skill gaps and provide an interactive dashboard to explore trends across industries, regions, and time periods.

In [11]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive

# Mount Drive
drive.mount("/content/drive", force_remount=True)

# Core paths
ROOT_DIR = Path("/content/drive/MyDrive/Career-Trends-Analyzer").resolve()
DATA_DIR = ROOT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
NOTE_DIR = ROOT_DIR / "notebooks"
SRC_DIR = ROOT_DIR / "src"
REPORT_DIR = ROOT_DIR / "report"

# Ensure folders
for p in (DATA_DIR, RAW_DIR, PROCESSED_DIR, NOTE_DIR, SRC_DIR, REPORT_DIR):
    p.mkdir(parents=True, exist_ok=True)

# Make src importable
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

# Make src a package
(SRC_DIR / "__init__.py").touch(exist_ok=True)

# Write shared config
config_code = f"""from pathlib import Path

ROOT_DIR = Path(r"{ROOT_DIR}").resolve()
DATA_DIR = ROOT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
NOTE_DIR = ROOT_DIR / "notebooks"
SRC_DIR = ROOT_DIR / "src"
REPORT_DIR = ROOT_DIR / "report"
"""
(SRC_DIR / "config.py").write_text(config_code, encoding="utf-8")

print("Setup OK")
print("ROOT_DIR:", ROOT_DIR)


Mounted at /content/drive
Setup OK
ROOT_DIR: /content/drive/MyDrive/Career-Trends-Analyzer


In [14]:


SRC_DIR = Path("/content/drive/MyDrive/Career-Trends-Analyzer/src")
data_loader_path = SRC_DIR / "data_loader.py"

data_loader_code = """from __future__ import annotations

from pathlib import Path
from typing import Dict

import pandas as pd

from .config import RAW_DIR

try:
    from google.colab import drive
except ImportError:  # pragma: no cover
    drive = None  # type: ignore[assignment]


def mount_drive() -> None:
    \"\"\"Mount Google Drive in Colab; no-op elsewhere.\"\"\"  # noqa: D401
    if drive is None:
        return
    drive.mount("/content/drive", force_remount=False)


def _load_csv(path: Path) -> pd.DataFrame:
    if not path.is_file():
        msg = f\"CSV not found: {path}\"
        raise FileNotFoundError(msg)
    return pd.read_csv(path)


def load_postings() -> pd.DataFrame:
    \"\"\"Load postings.csv from RAW_DIR.\"\"\"  # noqa: D401
    return _load_csv(RAW_DIR / "postings.csv")


def _load_folder(name: str) -> Dict[str, pd.DataFrame]:
    folder = RAW_DIR / name
    if not folder.is_dir():
        return {}
    return {p.stem: pd.read_csv(p) for p in folder.glob("*.csv")}


def load_companies() -> Dict[str, pd.DataFrame]:
    \"\"\"Load company tables from raw/companies.\"\"\"  # noqa: D401
    return _load_folder("companies")


def load_jobs() -> Dict[str, pd.DataFrame]:
    \"\"\"Load job tables from raw/jobs.\"\"\"  # noqa: D401
    return _load_folder("jobs")


def load_mappings() -> Dict[str, pd.DataFrame]:
    \"\"\"Load mapping tables from raw/mappings.\"\"\"  # noqa: D401
    return _load_folder("mappings")


def build_master(
    postings: pd.DataFrame,
    companies: Dict[str, pd.DataFrame],
    jobs: Dict[str, pd.DataFrame],
    mappings: Dict[str, pd.DataFrame],
) -> pd.DataFrame:
    \"\"\"Build a simple master table from safe joins.\"\"\"  # noqa: D401
    df = postings.copy()

    companies_df = companies.get("companies")
    if companies_df is not None and not companies_df.empty:
        if "company_id" in df.columns and "company_id" in companies_df.columns:
            df = df.merge(
                companies_df,
                on="company_id",
                how="left",
                suffixes=("", "_company"),
            )

    salaries_df = jobs.get("salaries")
    if salaries_df is not None and not salaries_df.empty:
        if "job_id" in df.columns and "job_id" in salaries_df.columns:
            df = df.merge(
                salaries_df,
                on="job_id",
                how="left",
                suffixes=("", "_salary"),
            )

    return df
"""

data_loader_path.write_text(data_loader_code, encoding="utf-8")
print(" data_loader.py written:", data_loader_path)


 data_loader.py written: /content/drive/MyDrive/Career-Trends-Analyzer/src/data_loader.py
